# Experiment 2: Qwen single shot (zero shot)

Enable **Internet** and a **GPU accelerator** in Kaggle settings. Commit and push `experiment2` to GitHub before using this notebook. Run the cells in order. Source, dependencies, and data are prepared automatically; credentials stay in Kaggle.

FP16 weights with efficient SDPA, full RGB images, original processor settings, greedy decoding, and strict JSON evaluation. Both Qwen modes exclude reference subject 0 from evaluation.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/ethanmachado2/CS297-research-project-em.git"
REF = "main"  # Set a published branch/tag/commit to freeze the source version.
REPO_DIR = Path("/kaggle/working/CS297-research-project-em")
PROJECT = REPO_DIR / "experiment2"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--sparse", REPOSITORY, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REF], check=True)
# Refuse to overwrite edits to tracked files in the runtime checkout.
if subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
    raise RuntimeError("Runtime checkout has edits; save them before changing REF.")
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "experiment2"], check=True)
if not (PROJECT / "requirements.txt").is_file():
    raise RuntimeError("Selected Git ref does not contain experiment2. Commit and push first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
# Keep regenerable caches separate from results. The worker imports torch only
# after selecting its GPU; this notebook never holds a model in GPU memory.
os.environ["HF_HUB_CACHE"] = "/kaggle/working/cs297_cache/huggingface/hub"
print("Source commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())


## Run settings

Start with `"smoke"` to validate a new runtime, then change to `"full"`. Each execution creates a fresh result directory; completed runs are preserved.


In [ ]:
SELECTION = "full"  # "smoke": fixed 12 images; "largest": largest input; "full": all 196 targets.
GPU = 0  # Use 1 for the second T4, if present.
DATASET_ROOT = None  # Optional attached/downloaded dataset; otherwise KaggleHub downloads or reuses it.


In [ ]:
command = [sys.executable, "-u", "-m", "src.run", "qwen_single_shot",
           "--selection", SELECTION, "--gpu", str(GPU)]
if DATASET_ROOT is not None:
    command += ["--dataset-root", str(DATASET_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)


## Results

Review both valid-response accuracy and **Correct / planned**. Invalid JSON, including Markdown fences, remains a failure.


In [ ]:
import json
import pandas as pd
from IPython.display import display

candidates = sorted((PROJECT / "outputs").glob("qwen_single_shot_" + SELECTION + "_*/run.json"))
if not candidates:
    raise RuntimeError("No run found; execute the workload cell first.")
latest = candidates[-1]
status = json.loads(latest.read_text())
print(json.dumps(status, indent=2))
if status.get("results_directory"):
    results = Path(status["results_directory"])
    display(pd.read_csv(results / "evaluation_summary.csv"))
    display(pd.read_csv(results / "coverage_by_severity.csv"))
    failures = pd.read_csv(results / "failures.csv")
    if not failures.empty:
        display(failures)


## Save artifacts

Generated data and outputs are gitignored. Preserve the result archive or save a Kaggle version with outputs before stopping the runtime. The archive includes all runs currently in this checkout.


In [ ]:
import shutil
from IPython.display import FileLink, display
archive = shutil.make_archive("/kaggle/working/qwen_single_shot_outputs", "zip", PROJECT, "outputs")
print("Download this archive or save a Kaggle notebook version with outputs before ending the session:", archive)
display(FileLink(archive))
